# Создание модели нейронной сети средствами Tensorflow

Библиотека <code>TensorFlow</code> предоставляет очень широкие возможности для работы с нейронными сетями. Ознакомиться со всеми возможностями можно заглянув в [официальную документацию](https://www.tensorflow.org/tutorials?hl=ru). В рамках данной же работы мы рассмотрим лишь некоторый функционал для работы с полносвязными НС.

Импортируем библиотеки

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import tensorflow as tf

# Немного про тензоры

С точки зрения `TF`, тензор — это многомерный массив. Подобно объектам `ndarray NumPy`, объекты `tf.Tensor` характеризуются типом данных и формой. В `TensorFlow` предусмотрен некоторый список готовых операций над тензорами (например, `tf.math.add`, `tf.linalg.matmul` и `tf.linalg.inv`). Эти операции автоматически преобразуют встроенные типы Python.

In [ ]:
print(tf.math.add(1, 2))
print(tf.math.add([1, 2], [3, 4]))
print(tf.math.square(5))

Преобразование `tf` ⟶ `np` и обратно происходит бесшовно

In [ ]:
ndarray = np.ones([3, 3])
tensor = tf.math.reduce_sum(ndarray, [0,1])
print(tensor)

In [ ]:
np.add(tensor, 1)

## Типы тензоров

`tf.constant()`: Создает тензор с фиксированными значениями. Подходит, когда нужно создать тензор с известными данными.

In [ ]:
tensor1 = tf.constant([1, 2, 3], dtype=tf.int32) # Целочисленный тензор
tensor2 = tf.constant([[1.0, 2.0], [3.0, 4.0]], dtype=tf.float32) # Тензор с плавающей точкой
tensor3 = tf.constant("Hello, TensorFlow!", dtype=tf.string) # Строковый тензор

print(tensor1)
print(tensor2)
print(tensor3)

`tf.Variable()`: Создает тензор, который можно изменять. Обычно используется для хранения весов и смещений в моделях.

In [ ]:
variable1 = tf.Variable([1.0, 2.0], dtype=tf.float32)
print(variable1)
variable1.assign([3.0, 4.0]) # Изменение значения переменной
print(variable1)

Кроме того, из названий понятно, для чего используются следующие типы тензоров:


*   tf.zeros()
*   tf.ones()
*   tf.random.normal()
*   tf.random.uniform()
*   tf.convert_to_tensor()



# Чтение и предварительная обработка данных

В качестве набора данных рассмотрим [следующий датасет.](https://raw.githubusercontent.com/rmcelreath/rethinking/master/data/Howell1.csv)

Это демографические данные жителей пустыни Калахари (охотники-собиратели), собранные Нэнси Хауэлл в Ботсване в период с августа 1967 по май 1969 года.

Считаем данные. Отберем относительно "стандартных" людей в возрасте от 18 до 50 лет.  

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/rmcelreath/rethinking/master/data/Howell1.csv', sep=';')
df_sample = df[(df['age'] >= 18) & (df['age'] <= 50)]
df_sample.head()

В качестве признаков будем рассматривать рост (<code>height</code>) и вес (<code>weight</code>), а предсказывать будем пол (<code>male</code>).

Визуализируем данные

In [ ]:
def plot(df):
    plt.figure(figsize=(8,6))
    plt.scatter(df.weight[df.male == 1], df.height[df.male == 1], color='blue', label='male')
    plt.scatter(df.weight[df.male == 0], df.height[df.male == 0], color='red', label='female')
    plt.legend()
    plt.ylabel('рост')
    plt.xlabel('масса')
plot(df_sample)

Разделим набор данных на тренировочную и тестовую части, а также зафиксируем <code>random_state</code> для воспроизводимости результатов.

In [ ]:
random_state = 42

X_train, X_test, y_train, y_test = train_test_split(df_sample[['weight', 'height']], df_sample['male'], test_size=0.2, random_state=random_state)

Приведем данные к необходимому формату. Во-первых `TF`привередлив к типам данных, во-вторых, отклики как одномерный массив "не катят", иначе будут проблемы (попробуйте убрать `.reshape(-1, 1)` для откликов).

In [ ]:
X_train = X_train.to_numpy(dtype=np.float32)
y_train = y_train.to_numpy(dtype=np.float32).reshape(-1, 1)
X_test = X_test.to_numpy(dtype=np.float32)
y_test = y_test.to_numpy(dtype=np.float32).reshape(-1, 1)

Раз мы имеем дело с нейронной сетью, имеет смысл нормировать данные.

In [ ]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)
X_test[:5]

# Проектирование модели

Зададим гиперпараметры

In [ ]:
# Количество входных признаков
input_size = 2
# Количество выходных классов (бинарная классификация)
output_size = 1
# Размер батча
batch_size = len(X_train)
# Скорость обучения
learning_rate = 0.5
# Количество эпох
epochs = 1000

Инициализируем переменные для весов и смещения

In [ ]:
tf.random.set_seed(random_state)
W = tf.Variable(tf.random.normal(shape=(input_size, output_size), dtype=tf.float32))
b = tf.Variable(tf.zeros(shape=(output_size,), dtype=tf.float32))

Функция активации

In [ ]:
def sigmoid(x):
    return 1 / (1 + tf.exp(-x))

Определим лосс

In [ ]:
def BCE(y_true, y_pred):
    return -tf.reduce_mean(y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))

Прямой проход

In [ ]:
def forward_pass(X):
    linear_output = tf.matmul(X, W) + b
    return sigmoid(linear_output)

Шаг обучения

In [ ]:
def train_step(X, y):
    with tf.GradientTape() as tape:
        # Прямой проход
        y_pred = forward_pass(X)
        # Вычисление потерь
        loss = BCE(y, y_pred)

    # Вычисление градиентов
    gradients = tape.gradient(loss, [W, b])

    # Обновление весов и смещений
    W.assign_sub(learning_rate * gradients[0])
    b.assign_sub(learning_rate * gradients[1])

    return loss


*   `tf.GradientTape()`: Это контекстный менеджер, который "записывает" все операции, выполненные внутри него, для последующего вычисления градиентов.
*   `tape.gradient(loss, [W, b])`: Вычисляет градиенты функции потерь loss относительно переменных W (веса) и b (смещение).
*   `W.assign_sub(learning_rate * gradients[0])` и `b.assign_sub()`: Обновляют значения весов и смещений с использованием градиентного спуска. assign_sub вычитает указанное значение из текущего значения переменной.
*   `tf.data.Dataset`: Используется для создания итерируемого объекта из данных, который позволяет эффективно обрабатывать большие наборы данных, разделяя их на пакеты.



Цикл обучения

In [ ]:
# Создание набора данных TensorFlow
dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(batch_size)

history = []

# Цикл по эпохам
for epoch in range(epochs):
    epoch_loss = 0.0
    # Цикл по пакетам данных
    for X_batch, y_batch in dataset:
        loss = train_step(X_batch, y_batch)
        epoch_loss += loss.numpy()
    history.append(epoch_loss)
    # Вывод информации о прогрессе обучения
    if epoch % int(epochs/10) == 0:
      print(f"Эпоха {epoch+1}, Loss: {epoch_loss / len(dataset)}")

In [ ]:
plt.plot(range(len(history)), history)

In [ ]:
W

In [ ]:
b

Оценка модели

In [ ]:
def predict(X):
  return np.array([1 if _ > 0.5 else 0 for _ in forward_pass(X)])

y_pred = predict(X_test)
f1_score(y_test, y_pred)

# Больше коробочных решений

Очевидно, что в `TF` уже реализованы многие компоненты, которые мы до этого писали "руками".

In [ ]:
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.losses import BinaryCrossentropy

Функция активации

In [ ]:
def sigmoid(x):
    return tf.nn.sigmoid(x)

Лосс

In [ ]:
BCE = BinaryCrossentropy()

Оптимизатор

In [ ]:
optimizer = SGD(learning_rate=learning_rate)

In [ ]:
def forward_pass(X):
    linear_output = tf.matmul(X, W) + b
    return sigmoid(linear_output)

Собираем все вместе + используем декоратор `@tf.function`. Это позволяет компилировать Python-функцию в graph `TensorFlow`, что значительно ускоряет выполнение кода, особенно на GPU и TPU.

In [ ]:
@tf.function
def train_step(X, y):
    with tf.GradientTape() as tape:
        y_pred = forward_pass(X)
        loss = BCE(y, y_pred)

    gradients = tape.gradient(loss, [W, b])

    # Обновление весов и смещений (теперь через оптимизатор)
    optimizer.apply_gradients(zip(gradients, [W, b]))

    return loss

In [ ]:
tf.random.set_seed(random_state)
W = tf.Variable(tf.random.normal(shape=(input_size, output_size), dtype=tf.float32))
b = tf.Variable(tf.zeros(shape=(output_size,), dtype=tf.float32))

In [ ]:
# Создание набора данных TensorFlow
dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(batch_size)

history = []

# Цикл по эпохам
for epoch in range(epochs):
    epoch_loss = 0.0
    # Цикл по пакетам данных
    for X_batch, y_batch in dataset:
        loss = train_step(X_batch, y_batch)
        epoch_loss += loss.numpy()
    history.append(epoch_loss)
    # Вывод информации о прогрессе обучения
    if epoch % int(epochs/10) == 0:
      print(f"Эпоха {epoch+1}, Loss: {epoch_loss / len(dataset)}")

In [ ]:
plt.plot(range(len(history)), history)

In [ ]:
W

In [ ]:
def predict(X):
  return np.array([1 if _ > 0.5 else 0 for _ in forward_pass(X)])

y_pred = predict(X_test)
f1_score(y_test, y_pred)

# Keras

## Небольшое описание

Раньше `Keras` был самостоятельной библиотекой для работы с НС на очень высоком уровне абстракции (в тч с `TF`), но затем стал частью `TF`. `Keras` предназначен для упрощения разработки моделей и решения повседневных (для нейросетей) задач. В случае исследовательских работ или построения сложных схем лучше делать это на уровне `TensorFlow` — с использованием тензоров, самостоятельно определяя слои и способы их соединения. `Keras` же помогает быстро построить модель и обучить ее, используя одну из заранее составленных архитектур, или построив простую архитектуру самостоятельно.






In [ ]:
from tensorflow import keras

На высоком уровне абстрации, рецепт для "приготовления" нейронной сети достаточно прост: нужны следующие ингридиенты:


*   **Архитектура** (слои и функции активации в них)
*   **Лосс**
*   **Оптимизатор**
*   **Метрика** (опционально)

![Логотип Python](https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/SMILE_data/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F%20image.png)

Давайте посмотрим, где и как в пакете `Keras` расположены необходимые нам элементы.
К текущему моменту мы пока что рассмотрили только один тип слоев — полносвязный, однако в дальнейшей в рамках курса мы с вами изучим и другие. Все слои располагаются в модуле `layers`.

In [ ]:
from tensorflow.keras import layers

Кроме того, мы с вами изучили некоторые функции активации, которые можно использовать при построении нейронных сетей. Все функции активации расположены в пакете `activations`.

In [ ]:
from tensorflow.keras import activations

Также импортируем пакет `optimizers`, в котором расположены оптимизаторы для обучения.

In [ ]:
from tensorflow.keras import optimizers

В модуле `losses` расположены различные функции потерь, которые можно использовать для обучения.

In [ ]:
from tensorflow.keras import losses

И последнее, что нам понадобится импортировать — это метрики из пакета `metrics`. Функция потерь — это некоторая функция, которая возвращает абстрактный `loss`, по которому сложно и зачастую совершенно невозможно оценить качество модели. Для измерения метрик, понятных людям, воспользуемся модулем `metrics` и заданными там функциями.

In [ ]:
from tensorflow.keras import metrics

## Вернемся к модели

**Модель**:

*   Архитектура:
    +   Входной слой
    +   1 нейрон, функция активации -- сигмоида
*   Лосс: BCE
*   Оптимизатор: Градиентный спуск
*   Метрика: f1_score

In [ ]:
# Тип модели: последовательная
model = keras.Sequential()

In [ ]:
# Добавляем входной слой
model.add(layers.Input(shape=X_train.shape[1:]))

In [ ]:
# Добавляем слой и функцию активации
model.add(layers.Dense(1, activation=activations.sigmoid))

In [ ]:
# Оптимизатор, лосс, метрика
model.compile(
    optimizer = optimizers.SGD(learning_rate = 0.5),
    loss = losses.BinaryCrossentropy(),
    metrics = [metrics.F1Score()]
)

In [ ]:
model.summary()

Обучение

In [ ]:
history = model.fit(X_train, y_train, epochs=1000, batch_size=len(X_train), verbose=0)

Отследить обучение можно по `history`

In [ ]:
history.history.keys()

In [ ]:
plt.plot(range(len(history.history['loss'])), history.history['loss'])

### Более краткая запись

In [ ]:
# Архитектура

model_short = keras.Sequential([
    layers.Input(shape=X_train.shape[1:]),
    layers.Dense(1, activation='sigmoid')
])

In [ ]:
# Оптимизатор, лосс, метрика

model_short.compile(optimizer='sgd',
              loss='binary_crossentropy',
              metrics=['f1_score'])

In [ ]:
model_short.summary()

### Предсказания и оценка

In [ ]:
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)
f1_score(y_test, y_pred)

Можно также посмотреть на сами веса

In [ ]:
model.layers[0].weights

## Многослойная НС

Аналогично рассмотренному выше создайте теперь многослойную сеть со следующей структурой скрытых слоев (число нейронов): `(256, 128, 64)`.

*   Функция активации в скрытых слоях — `relu`.
*   Активация выходного слоя `sigmoid`
*   Лосс — `binary_crossentropy`
*   Оптимизатор — стохастический градиентный спуск (`batch_size=32`)
*   Число эпох обучения — `100`

Обучите модель на тренировочных данных, оцените на тестовых. Удалось ли существенно увеличить значение метрики?


In [ ]:
# <ENTER YOUR CODE HERE>

Добавим валидацию. Для этого в метод `.fit()` добавьте параметр `validation_split=0.2`. Обучите модель, постройте графики лосса для тринировочного и валидационного набор данных (искать в `history`).

In [ ]:
# <ENTER YOUR CODE HERE>

Исследуйте варианты с более сложной/простой архитектурой, и другими  гиперпараметрами.

In [ ]:
# <ENTER YOUR CODE HERE>